# Solve Business Problems with AI

## Objective
Develop a proof-of-concept application to intelligently process email order requests and customer inquiries for a fashion store. The system should accurately categorize emails as either product inquiries or order requests and generate appropriate responses using the product catalog information and current stock status.

You are encouraged to use AI assistants (like ChatGPT or Claude) and any IDE of your choice to develop your solution. Many modern IDEs (such as PyCharm, or Cursor) can work with Jupiter files directly.

## Task Description

### Inputs

Google Spreadsheet **[Document](https://docs.google.com/spreadsheets/d/14fKHsblfqZfWj3iAaM2oA51TlYfQlFT4WKo52fVaQ9U)** containing:

- **Products**: List of products with fields including product ID, name, category, stock amount, detailed description, and season.

- **Emails**: Sequential list of emails with fields such as email ID, subject, and body.

### Instructions

- Implement all requirements using advanced Large Language Models (LLMs) to handle complex tasks, process extensive data, and generate accurate outputs effectively.
- Use Retrieval-Augmented Generation (RAG) and vector store techniques where applicable to retrieve relevant information and generate responses.
- You are provided with a temporary OpenAI API key granting access to GPT-4o, which has a token quota. Use it wisely or use your own key if preferred.
- Address the requirements in the order listed. Review them in advance to develop a general implementation plan before starting.
- Your deliverables should include:
   - Code developed within this notebook.
   - A single spreadsheet containing results, organized across separate sheets.
   - Comments detailing your thought process.
- You may use additional libraries (e.g., langchain) to streamline the solution. Use libraries appropriately to align with best practices for AI and LLM tools.
- Use the most suitable AI techniques for each task. Note that solving tasks with traditional programming methods will not earn points, as this assessment evaluates your knowledge of LLM tools and best practices.

### Requirements

#### 1. Classify emails
    
Classify each email as either a _**"product inquiry"**_ or an _**"order request"**_. Ensure that the classification accurately reflects the intent of the email.

**Output**: Populate the **email-classification** sheet with columns: email ID, category.

#### 2. Process order requests
1.   Process orders
  - For each order request, verify product availability in stock.
  - If the order can be fulfilled, create a new order line with the status “created”.
  - If the order cannot be fulfilled due to insufficient stock, create a line with the status “out of stock” and include the requested quantity.
  - Update stock levels after processing each order.
  - Record each product request from the email.
  - **Output**: Populate the **order-status** sheet with columns: email ID, product ID, quantity, status (**_"created"_**, **_"out of stock"_**).

2.   Generate responses
  - Create response emails based on the order processing results:
      - If the order is fully processed, inform the customer and provide product details.
      - If the order cannot be fulfilled or is only partially fulfilled, explain the situation, specify the out-of-stock items, and suggest alternatives or options (e.g., waiting for restock).
  - Ensure the email tone is professional and production-ready.
  - **Output**: Populate the **order-response** sheet with columns: email ID, response.

#### 3. Handle product inquiry

Customers may ask general open questions.
  - Respond to product inquiries using relevant information from the product catalog.
  - Ensure your solution scales to handle a full catalog of over 100,000 products without exceeding token limits. Avoid including the entire catalog in the prompt.
  - **Output**: Populate the **inquiry-response** sheet with columns: email ID, response.

## Evaluation Criteria
- **Advanced AI Techniques**: The system should use Retrieval-Augmented Generation (RAG) and vector store techniques to retrieve relevant information from data sources and use it to respond to customer inquiries.
- **Tone Adaptation**: The AI should adapt its tone appropriately based on the context of the customer's inquiry. Responses should be informative and enhance the customer experience.
- **Code Completeness**: All functionalities outlined in the requirements must be fully implemented and operational as described.
- **Code Quality and Clarity**: The code should be well-organized, with clear logic and a structured approach. It should be easy to understand and maintain.
- **Presence of Expected Outputs**: All specified outputs must be correctly generated and saved in the appropriate sheets of the output spreadsheet. Ensure the format of each output matches the requirements—do not add extra columns or sheets.
- **Accuracy of Outputs**: The accuracy of the generated outputs is crucial and will significantly impact the evaluation of your submission.

We look forward to seeing your solution and your approach to solving real-world problems with AI technologies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Prerequisites

In [ ]:
%pip install langchain langchain-community langchain-openai faiss-cpu tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


### Configure OpenAI API Key.

In [ ]:
# Install the OpenAI Python package.
%pip install openai httpx==0.27.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.68.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.


**IMPORTANT: If you are going to use our custom API Key then make sure that you also use custom base URL as in example below. Otherwise it will not work.**

In [ ]:
# Code example of OpenAI communication

from openai import OpenAI

client = OpenAI(
    # In order to use provided API key, make sure that models you create point to this custom base URL.
    base_url='https://47v4us7kyypinfb5lcligtc3x40ygqbs.lambda-url.us-east-1.on.aws/v1/',
    # The temporary API key giving access to ChatGPT 4o model. Quotas apply: you have 500'000 input and 500'000 output tokens, use them wisely ;)
    api_key='a0Bfv00000C9t9kEAB'
)

completion = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "user", "content": "Hello!"}
  ]
)

print(completion.choices[0].message)

ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [ ]:
# Code example of reading input data

import pandas as pd
from IPython.display import display

def read_data_frame(document_id, sheet_name):
    export_link = f"https://docs.google.com/spreadsheets/d/{document_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
    return  pd.read_csv(export_link)

document_id = '14fKHsblfqZfWj3iAaM2oA51TlYfQlFT4WKo52fVaQ9U'
products_df = read_data_frame(document_id, 'products')
emails_df = read_data_frame(document_id, 'emails')

# Display first 3 rows of each DataFrame
display(products_df.head(3))
display(emails_df.head(3))

,product_id,name,category,description,stock,seasons,price
0,RSG8901,Retro Sunglasses,Accessories,Transport yourself back in time with our retro...,1,"Spring, Summer",26.99
1,SWL2345,Sleek Wallet,Accessories,Keep your essentials organized and secure with...,5,All seasons,30.00
2,VSC6789,Versatile Scarf,Accessories,Add a touch of versatility to your wardrobe wi...,6,"Spring, Fall",23.00


,email_id,subject,message
0,E001,Leather Wallets,"Hi there, I want to order all the remaining LT..."
1,E002,Buy Vibrant Tote with noise,"Good morning, I'm looking to buy the VBT2345 V..."
2,E003,Need your help,"Hello, I need a new bag to carry my laptop and..."


In [ ]:
import json
import pandas as pd
import os
from openai import OpenAI
from dotenv import load_dotenv

# 1. Ensure API and data are loaded in this specific session
load_dotenv()
client = OpenAI(
    base_url='https://47v4us7kyypinfb5lcligtc3x40ygqbs.lambda-url.us-east-1.on.aws/v1/',
    api_key='a0Bfv00000C9t9kEAB'
)

# The emails_df is already loaded from the Google Sheet in a previous cell.
# The attempt to read 'emails.csv' is redundant and can cause a FileNotFoundError.
# Removing this block and relying on the emails_df loaded earlier.
# try:
#     emails_df = pd.read_csv("emails.csv")
#     print(f"Success: Loaded {len(emails_df)} emails for classification.\n")
# except FileNotFoundError:
#     print("Error: 'emails.csv' not found. Please ensure the file is in the same folder as this notebook.")

# 2. Define the classification function
def classify_email(subject, body):
    """Sends email content to GPT-4o for precise classification."""
    prompt = f"""
    You are an expert customer service assistant for a fashion store.
    Classify the following customer email into exactly one of these two categories:
    1. "product inquiry" - The customer is asking questions about sizing, materials, styles, availability, or general information.
    2. "order request" - The customer explicitly wants to buy, order, purchase, or modify an order for items.

    Email Subject: {subject}
    Email Body: {body}

    Respond ONLY with a valid JSON object containing a single key "category" with the value being either "product inquiry" or "order request".
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a precise classification assistant."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"}, # Enforces valid JSON response
            temperature=0.0 # Low temperature for consistent classification
        )

        # Parse the JSON response safely
        result = json.loads(response.choices[0].message.content)
        return result.get("category", "unknown")
    except Exception as e:
        print(f"Error classifying email: {e}")
        return "error"

# 3. Apply classification across the emails DataFrame
print("Classifying emails... please wait.")
classification_results = []

for idx, row in emails_df.iterrows():
    category = classify_email(row['subject'], row['message'])
    classification_results.append({
        "email ID": row['email_id'], # Changed to 'email_id' to match the loaded emails_df column name
        "category": category
    })

# 4. Convert to the required output DataFrame structure
email_classification_df = pd.DataFrame(classification_results)

print("\n--- Classification Complete ---")
display(email_classification_df.head(5))

Classifying emails... please wait.

--- Classification Complete ---


,email ID,category
0,E001,order request
1,E002,order request
2,E003,product inquiry
3,E004,order request
4,E005,product inquiry


In [ ]:
import json
import pandas as pd
import numpy as np
import re
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from openai import OpenAI

# --- Start of Added Code for DataFrames and Classification ---
def read_data_frame(document_id, sheet_name):
    export_link = f"https://docs.google.com/spreadsheets/d/{document_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
    return pd.read_csv(export_link)

document_id = '14fKHsblfqZfWj3iAaM2oA51TlYfQlFT4WKo52fVaQ9U'
products_df = read_data_frame(document_id, 'products')
emails_df = read_data_frame(document_id, 'emails')

client = OpenAI(
    base_url='https://47v4us7kyypinfb5lcligtc3x40ygqbs.lambda-url.us-east-1.on.aws/v1/',
    api_key='a0Bfv00000C9t9kEAB'
)

# Define the classification function
def classify_email(subject, body):
    """Sends email content to GPT-4o for precise classification."""
    prompt = f"""
    You are an expert customer service assistant for a fashion store.
    Classify the following customer email into exactly one of these two categories:
    1. "product inquiry" - The customer is asking questions about sizing, materials, styles, availability, or general information.
    2. "order request" - The customer explicitly wants to buy, order, purchase, or modify an order for items.

    Email Subject: {subject}
    Email Body: {body}

    Respond ONLY with a valid JSON object containing a single key "category" with the value being either "product inquiry" or "order request".
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a precise classification assistant."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.0
        )
        result = json.loads(response.choices[0].message.content)
        return result.get("category", "unknown")
    except Exception as e:
        print(f"Error classifying email: {e}")
        return "error"

# Apply classification
print("Classifying emails for order processing...")
classification_results = []
for idx, row in emails_df.iterrows():
    subj = row.get('subject', '')
    body = row.get('body', row.get('message', ''))
    email_identifier = row.get('email ID', row.get('email_id'))

    category = classify_email(subj, body)
    classification_results.append({
        "email ID": email_identifier,
        "category": category
    })

email_classification_df = pd.DataFrame(classification_results)
print("Email classification complete.\n")
# --- End of Added Code for DataFrames and Classification ---

# 1. Clean Products DataFrame for internal processing
products_df.columns = [c.strip().lower() for c in products_df.columns]
products_df.rename(columns={'id': 'product_id', 'product id': 'product_id', 'stock amount': 'stock'}, inplace=True, errors='ignore')

# Ensure internal IDs are strings for matching
products_df['product_id'] = products_df['product_id'].astype(str)

print("Building Product Vector Store for semantic search...")
product_texts = []
product_metadatas = []

for idx, row in products_df.iterrows():
    text = f"Product ID: {row['product_id']}. Name: {row.get('name', '')}. Category: {row.get('category', '')}. Description: {row.get('detailed description', row.get('description', ''))} {row.get('season', '')}"
    product_texts.append(text)
    product_metadatas.append({"product_id": row['product_id']})

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key='a0Bfv00000C9t9kEAB',
    openai_api_base='https://47v4us7kyypinfb5lcligtc3x40ygqbs.lambda-url.us-east-1.on.aws/v1/'
)
vector_store = FAISS.from_texts(product_texts, embeddings, metadatas=product_metadatas)
print("Vector store built successfully!\n")

# 2. Define the optimized extraction function
def extract_order_items(message_body):
    """Extracts items using strict JSON, isolated to current intentions."""
    prompt = f"""
    Extract ONLY the products and quantities the customer explicitly wants to order/buy right now from this email.
    Do NOT extract items they mention buying in the past, or items they say they might buy 'next time'.

    Email: "{message_body}"

    Respond ONLY with a JSON object containing a single key "items".
    The value of "items" must be an array of objects:
    - "product_id": string (the exact product ID if mentioned, leave any brackets or spaces included as written, e.g., "[CBT 89 01]")
    - "search_description": string (if NO exact ID/Name is given, write a rich 5-10 word search query. If an ID IS given, leave this empty "")
    - "quantity": integer (the specific number requested. Default to 1 if they just say they want to buy it. 0 if they ask for 'all')
    - "wants_all_stock": boolean (true ONLY if the customer explicitly asks for all remaining stock, otherwise false)
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.0
        )
        result = json.loads(response.choices[0].message.content)
        return result.get("items", [])
    except Exception as e:
        print(f"Extraction error: {e}")
        return []

# Helper function to remove spaces, brackets, dashes for clean evaluation
def normalize_string(s):
    return re.sub(r'[^a-zA-Z0-9]', '', str(s)).lower()

# 3. Process the Orders
print("Processing order requests with Resilient Normalization & Vector RAG...")
order_status_results = []

# Pre-calculate normalized IDs in the catalog to speed up matching
products_df['normalized_id'] = products_df['product_id'].apply(normalize_string)

for idx, row in email_classification_df.iterrows():
    email_id = row['email ID']
    category = row['category']

    if category == 'order request':
        email_row = emails_df[(emails_df.get('email ID') == email_id) | (emails_df.get('email_id') == email_id)]
        email_message = email_row.get('body', email_row.get('message', pd.Series(['']))).values[0]

        requested_items = extract_order_items(email_message)

        for item in requested_items:
            req_id_raw = item.get('product_id', '')
            req_id_normalized = normalize_string(req_id_raw)
            search_desc = item.get('search_description', '')
            req_qty = item.get('quantity', 1)  # Default to 1 if omitted by prompt
            wants_all_stock = item.get('wants_all_stock', False)

            match_id = None

            # Strategy A: Exact ID match (RESILIENT ALPHANUMERIC LOOKUP)
            if req_id_normalized:
                exact_match_df = products_df[products_df['normalized_id'] == req_id_normalized]
                if not exact_match_df.empty:
                    match_id = exact_match_df['product_id'].values[0]

            # Strategy B: RAG Semantic Search fallback
            if not match_id and search_desc:
                docs = vector_store.similarity_search(search_desc, k=1)
                if docs:
                    match_id = docs[0].metadata["product_id"]

            # Process inventory
            if match_id:
                prod_idx = products_df[products_df['product_id'] == match_id].index[0]
                current_stock = products_df.at[prod_idx, 'stock']

                if wants_all_stock:
                    req_qty = current_stock

                if current_stock >= req_qty and req_qty > 0:
                    products_df.at[prod_idx, 'stock'] = current_stock - req_qty
                    status = "created"
                else:
                    status = "out of stock"
            else:
                match_id = "UNKNOWN"
                status = "out of stock"

            # Strict alignment with grading instructions
            order_status_results.append({
                "email ID": email_id,
                "product ID": match_id,
                "quantity": req_qty,
                "status": status
            })

# Drop temporary column so products_df remains clean
products_df.drop(columns=['normalized_id'], inplace=True, errors='ignore')

order_status_df = pd.DataFrame(order_status_results)
print("\n--- Order Processing Complete ---")
display(order_status_df)

/tmp/ipykernel_2403/128095848.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Classifying emails for order processing...
Email classification complete.

Building Product Vector Store for semantic search...
Vector store built successfully!

Processing order requests with Resilient Normalization & Vector RAG...

--- Order Processing Complete ---


,email ID,product ID,quantity,status
0,E001,LTH0976,4,created
1,E002,VBT2345,1,created
2,E004,SFT1098,3,created
3,E007,CLF2109,5,out of stock
4,E007,FZZ1098,2,created
5,E008,VSC6789,1,created
6,E010,RSG8901,1,created
7,E013,SLD7654,1,created
8,E014,SWL2345,1,created
9,E017,CHK8901,1,created


Had to include Product ID clean up for formatting issues ie. E019
For E001, "all remaining" fix: needed to build an "escape hatch" into the prompt tol tell the AI to output the word "ALL" if a customer asks for everything. Then, update logic to intercept that word, look up the actual current_stock for that item, and set the requested quantity to match it exactly.

In [ ]:
import pandas as pd

print("Drafting order response emails...")
order_response_results = []

# Group the processed orders by email ID so we address all items in a single reply
grouped_orders = order_status_df.groupby('email ID')

def draft_order_response(customer_message, processing_results):
    """Uses GPT-4o to draft a professional response based on order fulfillment status."""
    prompt = f"""
    You are an expert customer service representative for a fashion store.
    Draft a professional, warm, and production-ready email reply to the customer based on their original message and the system's order processing results.

    Customer's Original Email:
    "{customer_message}"

    System Processing Results (JSON):
    {json.dumps(processing_results, indent=2)}

    Instructions for your response:
    - If all items have the status "created", inform the customer their order is fully confirmed and being processed and they will receive an additional email with tracking information as soon as the order ships. Provide brief product details.
    - If any item has the status "out of stock", politely explain the situation, specifically name the out-of-stock items, and suggest they wait for a restock or check out alternatives.
    - If the order is partially fulfilled, clearly separate what is confirmed from what is out of stock.
    - Do not include placeholders like [Your Name] or [Company Name]. Sign off generally as "The Customer Care Team".

    Respond ONLY with the final email text.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a professional fashion store customer service assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7 # Slightly higher temperature for natural, empathetic language
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Drafting error: {e}")
        return "Error generating response."

for email_id, group in grouped_orders:
    # Safely retrieve the original email text for context
    email_row = emails_df[(emails_df.get('email ID') == email_id) | (emails_df.get('email_id') == email_id)]
    email_message = email_row.get('body', email_row.get('message', pd.Series(['']))).values[0]

    # Convert the group's processing results into a list of dictionaries for the AI
    results_list = group[['product ID', 'quantity', 'status']].to_dict('records')

    # Draft the email
    drafted_reply = draft_order_response(email_message, results_list)

    # Strict alignment with rubric instructions
    order_response_results.append({
        "email ID": email_id,
        "response": drafted_reply
    })

order_response_df = pd.DataFrame(order_response_results)

print("\n--- Order Responses Complete ---")
# Displaying the first row's full response so you can read the generated text
display(order_response_df.head(2))
print("\nSample Email Output:\n")
print(order_response_df.iloc[0]['response'])

Drafting order response emails...

--- Order Responses Complete ---


,email ID,response
0,E001,Subject: Order Confirmation for LTH0976 Leathe...
1,E002,Subject: Your Order Confirmation for the Vibra...



Sample Email Output:

Subject: Order Confirmation for LTH0976 Leather Bifold Wallets

Dear [Customer's Name],

Thank you for reaching out to us and for your interest in our LTH0976 Leather Bifold Wallets. We are delighted to inform you that your order for all remaining units has been fully confirmed and is currently being processed!

We have created an order for the 4 LTH0976 Leather Bifold Wallets we have in stock. These wallets are crafted with premium leather, featuring a classic bifold design that combines style and functionality—perfect for your new boutique shop's inventory.

You will receive an additional email with tracking information once your order has been shipped. We are committed to ensuring your order arrives promptly and that you are completely satisfied with your purchase.

Should you have any further questions or need assistance, please feel free to contact us.

Thank you for choosing our products for your boutique, and we look forward to serving you again soon.

War

Included reference to follow-up email with tracking info when order is processed.

In [ ]:
import pandas as pd

print("Processing product inquiries using RAG...")
inquiry_response_results = []

def draft_inquiry_response(customer_message, retrieved_context):
    """Uses GPT-4o to answer product questions based on vector store context."""
    prompt = f"""
    You are an expert customer service representative for a fashion store.
    A customer has sent an inquiry about our products. Answer their questions clearly, warmly, and professionally using ONLY the relevant catalog information provided below.
    If the catalog info doesn't explicitly contain the answer to their specific question, politely let them know you don't have that exact information available right now.

    Relevant Catalog Information:
    {retrieved_context}

    Customer's Email:
    "{customer_message}"

    Respond ONLY with the final email text. Do not use placeholders like [Your Name]. Sign off as "The Customer Care Team".
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a professional fashion store customer service assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.5 # Kept moderate to balance professionalism with factual accuracy
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Drafting error: {e}")
        return "Error generating response."

for idx, row in email_classification_df.iterrows():
    email_id = row['email ID']
    category = row['category']

    if category == 'product inquiry':
        # Safely retrieve the original email text
        email_row = emails_df[(emails_df.get('email ID') == email_id) | (emails_df.get('email_id') == email_id)]
        email_message = email_row.get('body', email_row.get('message', pd.Series(['']))).values[0]

        # 1. RAG Retrieval: Find the top 3 most relevant products based on the customer's email
        docs = vector_store.similarity_search(email_message, k=3)

        # 2. Format the retrieved documents into a clean string for the LLM
        context_pieces = [doc.page_content for doc in docs]
        retrieved_context = "\n\n".join(context_pieces)

        # 3. Draft the email response
        drafted_reply = draft_inquiry_response(email_message, retrieved_context)

        # Strict alignment with grading instructions
        inquiry_response_results.append({
            "email ID": email_id,
            "response": drafted_reply
        })

inquiry_response_df = pd.DataFrame(inquiry_response_results)

print("\n--- Product Inquiry Responses Complete ---")
display(inquiry_response_df.head(2))

print("\nSample Inquiry Response:\n")
if not inquiry_response_df.empty:
    print(inquiry_response_df.iloc[0]['response'])

Processing product inquiries using RAG...

--- Product Inquiry Responses Complete ---


,email ID,response
0,E003,"Hello David,\n\nThank you for reaching out to ..."
1,E005,"Good day,\n\nThank you for reaching out to us ..."



Sample Inquiry Response:

Hello David,

Thank you for reaching out to us with your inquiry. We're delighted to assist you in choosing the perfect bag for your work needs.

Both the LTH1098 Leather Backpack and the Leather Tote are excellent choices, each offering unique features:

- The **LTH1098 Leather Backpack** is designed with multiple compartments and a padded laptop sleeve, making it ideal for organizing your laptop and documents. Its adjustable straps ensure a comfortable fit, perfect for work and travel.

- The **Leather Tote**, on the other hand, offers a spacious interior with multiple pockets, providing ample space and easy access to your essentials. Its sturdy handles make it a stylish choice for work, travel, or errands.

If organization and carrying your laptop securely are your top priorities, the Leather Backpack might be the better option due to its multiple compartments and padded laptop sleeve. However, if you prefer a more open design with easy access, the Leather

In [ ]:
!pip install gspread-dataframe

In [ ]:
from google.colab import auth
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

print("Authenticating with Google...")
# IMPORTANT: A pop-up will appear asking you to allow Colab to access your Google account.
auth.authenticate_user()
gcreds, _ = default()
gc = gspread.authorize(gcreds)

print("Creating final Output Document...")
# Create the Google Sheet
output_document = gc.create('Solving Business Problems with AI - Output')

# 1. Create and populate 'email-classification' sheet
email_classification_sheet = output_document.add_worksheet(title="email-classification", rows=100, cols=2)
set_with_dataframe(email_classification_sheet, email_classification_df)

# 2. Create and populate 'order-status' sheet
order_status_sheet = output_document.add_worksheet(title="order-status", rows=100, cols=4)
set_with_dataframe(order_status_sheet, order_status_df)

# 3. Create and populate 'order-response' sheet
order_response_sheet = output_document.add_worksheet(title="order-response", rows=100, cols=2)
set_with_dataframe(order_response_sheet, order_response_df)

# 4. Create and populate 'inquiry-response' sheet
inquiry_response_sheet = output_document.add_worksheet(title="inquiry-response", rows=100, cols=2)
set_with_dataframe(inquiry_response_sheet, inquiry_response_df)

# Clean up: Delete the default empty "Sheet1" that gets created automatically
try:
    worksheet_to_delete = output_document.worksheet("Sheet1")
    output_document.del_worksheet(worksheet_to_delete)
except:
    pass

# Share the spreadsheet publicly
output_document.share('', perm_type='anyone', role='reader')

print("\n🎉 SUCCESS! Here is your final submission link:")
print(f"Shareable link: https://docs.google.com/spreadsheets/d/{output_document.id}")

Authenticating with Google...
Creating final Output Document...

🎉 SUCCESS! Here is your final submission link:
Shareable link: https://docs.google.com/spreadsheets/d/1GwMZUJDH9b6kouqJRpFT2eo_p8L4L3SFCQw6Rkc5_aE


Included code to remove Sheet1 tab that gets generated with the output file.

In [ ]:
# DON'T USE THIS ONE- USE THE PREVIOUS CODE CELL AND LINK
# Code example of generating output document

# Creates a new shared Google Worksheet every invocation with the proper structure
# Note: This code should be executed from the google colab once you are ready, it will not work locally
# from google.colab import auth
# import gspread
# from google.auth import default
# from gspread_dataframe import set_with_dataframe

# IMPORTANT: You need to authenticate the user to be able to create new worksheet
# Insert the authentication snippet from the official documentation to create a google client:
# https://colab.research.google.com/notebooks/io.ipynb#scrollTo=qzi9VsEqzI-o

# auth.authenticate_user()
# gcreds, _ = default()
# gc = gspread.authorize(gcreds)

# # This code goes after creating google client
# output_document = gc.create('Solving Business Problems with AI - Output')

# # Create 'email-classification' sheet
# email_classification_sheet = output_document.add_worksheet(title="email-classification", rows=50, cols=2)
# email_classification_sheet.update([['email ID', 'category']], 'A1:B1')

# # Example of writing the data into the sheet
# # Assuming you have your classification in the email_classification_df DataFrame
# # set_with_dataframe(email_classification_sheet, email_classification_df)
# # Or directly update cells: https://docs.gspread.org/en/latest/user-guide.html#updating-cells

# # Create 'order-status' sheet
# order_status_sheet = output_document.add_worksheet(title="order-status", rows=50, cols=4)
# order_status_sheet.update([['email ID', 'product ID', 'quantity', 'status']], 'A1:D1')

# # Create 'order-response' sheet
# order_response_sheet = output_document.add_worksheet(title="order-response", rows=50, cols=2)
# order_response_sheet.update([['email ID', 'response']], 'A1:B1')

# # Create 'inquiry-response' sheet
# inquiry_response_sheet = output_document.add_worksheet(title="inquiry-response", rows=50, cols=2)
# inquiry_response_sheet.update([['email ID', 'response']], 'A1:B1')

# # Share the spreadsheet publicly
# output_document.share('', perm_type='anyone', role='reader')

# This is the solution output link, paste it into the submission form
# print(f"Shareable link: https://docs.google.com/spreadsheets/d/{output_document.id}")

Shareable link: https://docs.google.com/spreadsheets/d/17aGpeS24bgA4XvUdTcdJzcSCStt5S0sA2k4wB7AGgtw


# Task 1. Classify emails

# Task 2. Process order requests

# Task 3. Handle product inquiry